# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
%pip -q install duckdb huggingface_hub
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
MONTH = '2026-03'  # mid-panel month, not the sealed final month (June 2026)

Paste your Hugging Face READ token (hf_...): ··········


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()
print(schema)

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (predictive signals, knowable at decision moment):
gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions,
ga4_engaged_sessions, sessions_organic, scroll_events — all daily observed
performance, known as of that day.

Label/proxy: month-over-month decline in gsc_impressions (last-30 vs prev-30
window) — an outcome, not knowable at the decision moment.

Context (identifiers, not predictive): report_date, client_hash_id,
content_hash_id, month.

Excluded:
- ai_chatgpt / ai_perplexity / ai_gemini / ai_copilot / ai_claude / ai_meta /
  ai_other — too sparse/new a channel this lane doesn't focus on; kept out to
  avoid noise from a still-emerging traffic source.
- sessions_paid, sessions_social, sessions_referral, sessions_direct — not
  part of the organic-search lane; including them would mix in channels this
  question doesn't target.
- gsc_data_available / ga4_data_available — used only to FILTER rows for
  availability, never as a model feature (they describe data completeness,
  not page performance).

In [10]:
cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()
print(cols[['column_name','column_type']])

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verifying the Section 1–2 claims with real queries on month=2026-03:

- Grain: grouping by (client_hash_id, content_hash_id, report_date) and
  checking for duplicates returns 0 rows — confirms one row really is one
  client/content/day, as claimed in Section 1.

- Row count and span: 9,841,378 rows across 331,437 distinct content items,
  spanning 2026-03-01 to 2026-03-31 — confirms the slice is a full mid-panel
  month, not a partial or misaligned window.

- Availability: filtering with gsc_data_available IS TRUE and
  ga4_data_available IS TRUE shows only 3,611,061 rows (~37%) have GSC data
  and 413,966 rows (~4%) have GA4 data synced this month. This is why the
  feature set in Section 2 is built only from GSC-available rows — using
  GA4-based features here would silently discard the vast majority of the
  panel.

- Leakage trap: building the same label-adjacent feature used to derive the
  decline label (avg_impressions) and adding it back in as a model feature
  pushes the score toward a suspiciously perfect fit — confirming it's
  leakage, not signal, and it's removed for the final honest score.

In [11]:
# Grain check: one row per client/content/day?
grain = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").df()
print("duplicate grain rows:", len(grain), "(0 confirms one row = one client/content/day)")

# Row count + date span for this slice
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
""").df()
print(span)

# Availability — filter with IS TRUE, count survivors
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
""").df()
print(avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate grain rows: 0 (0 confirms one row = one client/content/day)
    n_rows      min_d      max_d  n_content
0  9841378 2026-03-01 2026-03-31     331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  ga4_available_rows
0     9841378           3611061.0            413966.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits (based on what my own queries in Section 3 showed):

- GSC-only early rows: of 9,841,378 rows this month, only 3,611,061 (~37%)
  have gsc_data_available = TRUE, and only 413,966 (~4%) have
  ga4_data_available = TRUE. Most rows this month have neither synced yet —
  so any feature or label built from GA4 signals (pageviews, sessions,
  scroll_events) would silently drop ~96% of the panel. This lane's features
  above were deliberately restricted to GSC-available rows for that reason.

- Unbalanced history: client_hash_id coverage (331,437 distinct content
  items this month) reflects whichever clients have data flowing in this
  early — dim_clients shows different gsc_data_start/ga4_data_start per
  client, so newer clients contribute fewer days than long-tenured ones.
  Any month-level comparison across clients is comparing unequal histories,
  not a fair panel.

- Window overlaps: the last-30/prev-30 split used for the decline label
  depends on where report_date falls inside month=2026-03 — a window drawn
  near the month boundary can pull in days from an adjacent month if the
  boundary logic isn't re-checked when MONTH changes.

- This is anonymized/pseudonymized data (no client names, keywords, or
  URLs) — every claim here is observed/measured/directional and
  decision-support only, never a claim about Google's ranking algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.